In [1]:
#Importing the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Loading one of the parquet files to check the data 
df = pd.read_parquet('../data/raw/trip_records/2023-1.parquet')

In [ ]:
# Inspecting the data
df.info()

<class 'pandas.DataFrame'>
Index: 3041714 entries, 0 to 3066765
Data columns (total 19 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int64         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     str           
 7   PULocationID           int64         
 8   DOLocationID           int64         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  airport_fee            float64       


In [10]:
import os

# List all parquet files
file_list = sorted([
    f for f in os.listdir('../data/raw/trip_records/') 
    if f.endswith('.parquet')
])

print(f"Found {len(file_list)} files: {file_list}")

# Empty dataframe to store all sampled data
df_stratified = pd.DataFrame()

# Iterate through each file (each month)
for file_name in file_list:
    try:
        # File path
        file_path = os.path.join('../data/raw/trip_records/', file_name)
        
        # Read the file
        df = pd.read_parquet(file_path)
        
        # Create date and hour columns
        df['Pickup_Date'] = df['tpep_pickup_datetime'].dt.date
        df['Pickup_Hour'] = df['tpep_pickup_datetime'].dt.hour
        
        # Empty dataframe for this month's sample
        sampled_data = pd.DataFrame()
        
        # Get all unique dates in this month
        dates = sorted(df['Pickup_Date'].unique())
        
        # Loop through each date
        for date in dates:
            
            # Filter data for current date
            date_data = df[df['Pickup_Date'] == date]
            
            # Loop through each hour (0 to 23)
            for h in range(24):
                
                # Filter for this hour
                hour_data = date_data[date_data['Pickup_Hour'] == h]
                
                # Only sample if data exists for this hour
                if not hour_data.empty:
                    sample = hour_data.sample(
                        frac=0.05, 
                        random_state=42
                    )
                    sampled_data = pd.concat(
                        [sampled_data, sample], 
                        ignore_index=True
                    )
        
        # Add this month to the full year sample
        df_stratified = pd.concat(
            [df_stratified, sampled_data], 
            ignore_index=True
        )
        
        print(f"{file_name}: {len(df):,} rows → {len(sampled_data):,} sampled")
    
    except Exception as e:
        print(f"Error reading file {file_name}: {e}")

# Final summary
print(f"\nFinal shape: {df_stratified.shape}")
print(f"Hours covered: {sorted(df_stratified['Pickup_Hour'].unique().tolist())}")
print(f"Months covered: {sorted(pd.to_datetime(df_stratified['tpep_pickup_datetime']).dt.month.unique().tolist())}")
print(f"Memory usage: {df_stratified.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Found 12 files: ['2023-1.parquet', '2023-10.parquet', '2023-11.parquet', '2023-12.parquet', '2023-2.parquet', '2023-3.parquet', '2023-4.parquet', '2023-5.parquet', '2023-6.parquet', '2023-7.parquet', '2023-8.parquet', '2023-9.parquet']
2023-1.parquet: 3,041,714 rows → 152,087 sampled
2023-10.parquet: 3,485,185 rows → 174,255 sampled
2023-11.parquet: 3,302,857 rows → 165,133 sampled
2023-12.parquet: 3,333,925 rows → 166,709 sampled
2023-2.parquet: 3,374,086 rows → 168,696 sampled
2023-3.parquet: 3,275,796 rows → 163,786 sampled
2023-4.parquet: 2,792,901 rows → 139,641 sampled
2023-5.parquet: 2,889,185 rows → 144,458 sampled
2023-6.parquet: 3,258,261 rows → 162,910 sampled
2023-7.parquet: 3,481,547 rows → 174,068 sampled
2023-8.parquet: 2,875,947 rows → 143,782 sampled
2023-9.parquet: 2,817,156 rows → 140,875 sampled

Final shape: (1896400, 22)
Hours covered: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
Months covered: [1, 2, 3, 4, 5, 6, 7, 8, 9,

In [12]:
max_entries = 300000

print("Stratified sample shape =", df_stratified.shape)

if len(df_stratified) > max_entries:
    df_final = df_stratified.sample(n=max_entries, random_state=42).reset_index(drop=True)
    df_final.to_parquet('../data/processed/nyc_taxi_300k.parquet', index=False)
    print("Final shape =", df_final.shape)
    print(f"Hours covered: {sorted(df_final['Pickup_Hour'].unique().tolist())}")
    print(f"Months covered: {sorted(df_final['tpep_pickup_datetime'].dt.month.unique().tolist())}")
    print(f"Memory usage: {df_final.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
    print("Saved: data/processed/nyc_taxi_300k.parquet")

Stratified sample shape = (1896400, 22)
Final shape = (300000, 22)
Hours covered: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
Months covered: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Memory usage: 58.7 MB
Saved: data/processed/nyc_taxi_300k.parquet
